# 11 — Unified GeoPackage Metadata and Publication Schema

## Objective

Notebook 09 established the canonical harmonized structure of the Canadian
geological CO₂ storage database, while Notebook 10 validated its logical
relational structure and explored whether additional SQL-level relationship
enforcement was necessary.

The purpose of Notebook 11 is to finalize the **metadata, provenance, and
GeoPackage registration model** for the unified database so that it can later
be reproduced as a standard output of the CanCO₂Repo workflow.

The underlying geological harmonization and four-table domain model are treated
as established:

- `storage_units`
- `storage_features`
- `storage_assessments`
- `administrative_features`

This notebook will therefore not redesign geological unit identity or introduce
additional relational structure unless a metadata or GeoPackage compliance
requirement demonstrates that it is necessary.

## Design principle

The unified GeoPackage should function as both:

1. a portable spatial database containing the harmonized Canadian geological
   CO₂ storage data; and
2. a self-describing research artifact containing enough metadata, provenance,
   field documentation, and quality-assurance information for another researcher
   to understand how the database was constructed and how its fields should be
   interpreted.

GeoPackage is a relational SQLite container, but not every logical domain
relationship must be represented through an additional GeoPackage relationship
extension. The canonical tables already separate conceptual storage units,
spatial representations, assessments, and administrative features.

Following the OGC GeoPackage modelling guidance, spatial representations should
remain in dedicated feature tables with one geometry column per feature table,
while associated non-spatial information can remain in related attribute
tables. The existing canonical structure already follows this pattern.

## Metadata objectives

Notebook 11 will:

1. inspect the metadata and documentation conventions used by the source Silver
   GeoPackages from which the unified database was derived;

2. define a standardized metadata model for the unified product, including:
   - dataset-level metadata;
   - source-dataset provenance;
   - table descriptions;
   - field definitions and units;
   - harmonization notes;
   - interpretation and use limitations;
   - quality-assurance summaries;

3. register the canonical tables consistently within the GeoPackage, including
   non-spatial canonical tables as GeoPackage attribute tables where appropriate;

4. populate standard GeoPackage metadata structures where they provide useful
   interoperability, including evaluation of:
   - `gpkg_contents`;
   - `gpkg_metadata`;
   - `gpkg_metadata_reference`;
   - `gpkg_data_columns`;
   - `gpkg_extensions`;

5. preserve provenance from NATCARB, the Northeast British Columbia Storage
   Atlas, the Atlantic Canada COS dataset, and Alberta carbon-sequestration
   agreement data without copying obsolete source-layer references into the
   harmonized database;

6. validate the completed GeoPackage as the proposed publication and exchange
   schema for later integration into CanCO₂Repo.

## Intended outcome

The final GeoPackage should allow a researcher to determine, from the database
itself:

- what each canonical table represents;
- what each field means and, where applicable, its units;
- which source dataset contributed each record;
- the citation, licence, and provenance associated with each source dataset;
- how the source datasets were harmonized;
- which limitations or interpretation rules apply to the data;
- what QA checks were performed; and
- which tables constitute spatial features versus non-spatial attributes.

The resulting schema will serve as the reference design for a later
programmatic CanCO₂Repo metadata and GeoPackage export workflow.

In [6]:
# ---------------------------------------------------------------------------
# Notebook setup
# ---------------------------------------------------------------------------

from pathlib import Path
import sqlite3

import geopandas as gpd
import pandas as pd


# ---------------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\aviga\Research\repos\canco2-storage"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


# ---------------------------------------------------------------------------
# Source Silver GeoPackages
# ---------------------------------------------------------------------------

SOURCE_GPKGS = {
    "aer_agreements": (
        PROCESSED_DIR
        / "aer_agreements"
        / "20260913_13_AERCarbonSequestrationAgreements_AV.gpkg"
    ),
    "gbc_ne_atlas": (
        PROCESSED_DIR
        / "gbc_ne_atlas"
        / "20260913_13_BCStorageAtlas_AV.gpkg"
    ),
    "gsc_atlantic": (
        PROCESSED_DIR
        / "gsc_atlantic"
        / "20260913_13_AtlanticStorageCOS_AV.gpkg"
    ),
    "natcarb_doe": (
        PROCESSED_DIR
        / "natcarb_doe"
        / "20260913_13_NATCARBStorage_AV.gpkg"
    ),
}


# ---------------------------------------------------------------------------
# Unified GeoPackage
# ---------------------------------------------------------------------------

GPKG_PATH = (
    PROCESSED_DIR
    / "unified_storage"
    / "canada_geological_storage_unified.gpkg"
)


# ---------------------------------------------------------------------------
# Canonical unified tables
# ---------------------------------------------------------------------------

CANONICAL_TABLES = [
    "storage_units",
    "storage_features",
    "storage_assessments",
    "administrative_features",
]


# ---------------------------------------------------------------------------
# Validate required files
# ---------------------------------------------------------------------------

print("Source Silver GeoPackages")
print("-------------------------")

for dataset_id, gpkg_path in SOURCE_GPKGS.items():
    status = "FOUND" if gpkg_path.is_file() else "MISSING"

    print(
        f"{dataset_id:<20} "
        f"{status:<8} "
        f"{gpkg_path}"
    )


print("\nUnified GeoPackage")
print("------------------")

unified_status = "FOUND" if GPKG_PATH.is_file() else "MISSING"

print(
    f"{unified_status:<8} "
    f"{GPKG_PATH}"
)


# ---------------------------------------------------------------------------
# Fail early if required inputs are unavailable
# ---------------------------------------------------------------------------

missing_sources = {
    dataset_id: path
    for dataset_id, path in SOURCE_GPKGS.items()
    if not path.is_file()
}

if missing_sources:
    raise FileNotFoundError(
        "Missing source Silver GeoPackages:\n"
        + "\n".join(
            f"- {dataset_id}: {path}"
            for dataset_id, path in missing_sources.items()
        )
    )

if not GPKG_PATH.is_file():
    raise FileNotFoundError(
        f"Unified GeoPackage not found: {GPKG_PATH}"
    )

Source Silver GeoPackages
-------------------------
aer_agreements       FOUND    C:\Users\aviga\Research\repos\canco2-storage\data\processed\aer_agreements\20260913_13_AERCarbonSequestrationAgreements_AV.gpkg
gbc_ne_atlas         FOUND    C:\Users\aviga\Research\repos\canco2-storage\data\processed\gbc_ne_atlas\20260913_13_BCStorageAtlas_AV.gpkg
gsc_atlantic         FOUND    C:\Users\aviga\Research\repos\canco2-storage\data\processed\gsc_atlantic\20260913_13_AtlanticStorageCOS_AV.gpkg
natcarb_doe          FOUND    C:\Users\aviga\Research\repos\canco2-storage\data\processed\natcarb_doe\20260913_13_NATCARBStorage_AV.gpkg

Unified GeoPackage
------------------
FOUND    C:\Users\aviga\Research\repos\canco2-storage\data\processed\unified_storage\canada_geological_storage_unified.gpkg


In [7]:
# ---------------------------------------------------------------------------
# Inspect the current unified GeoPackage structure
# ---------------------------------------------------------------------------

conn = sqlite3.connect(GPKG_PATH)

try:
    sqlite_objects = pd.read_sql_query(
        """
        SELECT
            name,
            type
        FROM sqlite_master
        WHERE type IN ('table', 'view')
        ORDER BY type, name;
        """,
        conn,
    )

    gpkg_contents = pd.read_sql_query(
        """
        SELECT *
        FROM gpkg_contents
        ORDER BY table_name;
        """,
        conn,
    )

    gpkg_geometry_columns = pd.read_sql_query(
        """
        SELECT *
        FROM gpkg_geometry_columns
        ORDER BY table_name;
        """,
        conn,
    )

    gpkg_extensions = pd.read_sql_query(
        """
        SELECT *
        FROM gpkg_extensions
        ORDER BY table_name, extension_name;
        """,
        conn,
    )

finally:
    conn.close()


# ---------------------------------------------------------------------------
# Validate canonical table presence
# ---------------------------------------------------------------------------

existing_tables = set(
    sqlite_objects.loc[
        sqlite_objects["type"] == "table",
        "name",
    ]
)

missing_canonical_tables = (
    set(CANONICAL_TABLES) - existing_tables
)

print("Canonical unified tables")
print("------------------------")

for table_name in CANONICAL_TABLES:
    status = "FOUND" if table_name in existing_tables else "MISSING"
    print(f"{table_name:<28} {status}")


if missing_canonical_tables:
    raise ValueError(
        "Unified GeoPackage is missing canonical tables: "
        f"{sorted(missing_canonical_tables)}"
    )


# ---------------------------------------------------------------------------
# Current GeoPackage registration
# ---------------------------------------------------------------------------

print("\nRegistered gpkg_contents")
print("------------------------")
display(gpkg_contents)

print("\nRegistered geometry columns")
print("---------------------------")
display(gpkg_geometry_columns)

print("\nRegistered extensions")
print("---------------------")
display(gpkg_extensions)


# ---------------------------------------------------------------------------
# Identify canonical tables not registered in gpkg_contents
# ---------------------------------------------------------------------------

registered_contents = set(gpkg_contents["table_name"])

unregistered_canonical_tables = (
    set(CANONICAL_TABLES) - registered_contents
)

print("\nCanonical tables not registered in gpkg_contents")
print("------------------------------------------------")

if unregistered_canonical_tables:
    for table_name in sorted(unregistered_canonical_tables):
        print(f"- {table_name}")
else:
    print("None")

Canonical unified tables
------------------------
storage_units                FOUND
storage_features             FOUND
storage_assessments          FOUND
administrative_features      FOUND

Registered gpkg_contents
------------------------


,table_name,data_type,identifier,description,last_change,min_x,min_y,max_x,max_y,srs_id
0,administrative_features,features,administrative_features,,2026-09-16T22:16:57.241Z,-1.496583e+06,212143.324821,-9.416869e+05,1.059745e+06,3978
1,storage_features,features,storage_features,,2026-09-16T22:16:57.106Z,-2.349109e+06,-706608.096292,3.049300e+06,2.934766e+06,3978



Registered geometry columns
---------------------------


,table_name,column_name,geometry_type_name,srs_id,z,m
0,administrative_features,geom,MULTIPOLYGON,3978,0,0
1,storage_features,geom,GEOMETRY,3978,0,0



Registered extensions
---------------------


,table_name,column_name,extension_name,definition,scope
0,administrative_features,geom,gpkg_rtree_index,http://www.geopackage.org/spec120/#extension_r...,write-only
1,storage_features,geom,gpkg_rtree_index,http://www.geopackage.org/spec120/#extension_r...,write-only



Canonical tables not registered in gpkg_contents
------------------------------------------------
- storage_assessments
- storage_units


In [8]:
# ---------------------------------------------------------------------------
# Inspect precursor metadata and QA tables
# ---------------------------------------------------------------------------

precursor_doc_tables = []
precursor_doc_schemas = []
precursor_doc_contents = []

for dataset_id, gpkg_path in SOURCE_GPKGS.items():

    conn = sqlite3.connect(gpkg_path)

    try:
        # -------------------------------------------------------------------
        # Registered GeoPackage contents
        # -------------------------------------------------------------------

        gpkg_contents = pd.read_sql_query(
            """
            SELECT
                table_name,
                data_type,
                identifier,
                description,
                srs_id
            FROM gpkg_contents
            ORDER BY table_name;
            """,
            conn,
        )

        registered_lookup = dict(
            zip(
                gpkg_contents["table_name"],
                gpkg_contents["data_type"],
            )
        )

        # -------------------------------------------------------------------
        # Find metadata / QA tables
        # -------------------------------------------------------------------

        user_tables = pd.read_sql_query(
            """
            SELECT
                name AS table_name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
              AND name NOT LIKE 'gpkg_%'
              AND name NOT LIKE 'rtree_%'
            ORDER BY name;
            """,
            conn,
        )

        doc_tables = [
            table_name
            for table_name in user_tables["table_name"]
            if (
                table_name.lower().startswith("metadata_")
                or table_name.lower().startswith("qa_")
            )
        ]

        # -------------------------------------------------------------------
        # Inspect each documentation table
        # -------------------------------------------------------------------

        for table_name in doc_tables:

            quoted_table = '"' + table_name.replace('"', '""') + '"'

            row_count = conn.execute(
                f"SELECT COUNT(*) FROM {quoted_table};"
            ).fetchone()[0]

            precursor_doc_tables.append(
                {
                    "dataset_id": dataset_id,
                    "table_name": table_name,
                    "gpkg_data_type": registered_lookup.get(table_name),
                    "row_count": row_count,
                }
            )

            # ---------------------------------------------------------------
            # Schema
            # ---------------------------------------------------------------

            schema = pd.read_sql_query(
                f"PRAGMA table_info({quoted_table});",
                conn,
            )

            schema.insert(0, "dataset_id", dataset_id)
            schema.insert(1, "table_name", table_name)

            precursor_doc_schemas.append(schema)

            # ---------------------------------------------------------------
            # Contents
            # ---------------------------------------------------------------

            contents = pd.read_sql_query(
                f"SELECT * FROM {quoted_table};",
                conn,
            )

            contents.insert(0, "dataset_id", dataset_id)
            contents.insert(1, "table_name", table_name)

            precursor_doc_contents.append(contents)

    finally:
        conn.close()


# ---------------------------------------------------------------------------
# Combine results
# ---------------------------------------------------------------------------

precursor_doc_tables = pd.DataFrame(
    precursor_doc_tables
).sort_values(
    ["dataset_id", "table_name"]
).reset_index(drop=True)


precursor_doc_schemas = pd.concat(
    precursor_doc_schemas,
    ignore_index=True,
)


precursor_doc_contents = pd.concat(
    precursor_doc_contents,
    ignore_index=True,
)


# ---------------------------------------------------------------------------
# Display table inventory
# ---------------------------------------------------------------------------

print("Precursor metadata and QA table inventory")
print("-----------------------------------------")

display(precursor_doc_tables)


# ---------------------------------------------------------------------------
# Display schemas
# ---------------------------------------------------------------------------

print("\nPrecursor metadata and QA schemas")
print("---------------------------------")

display(
    precursor_doc_schemas[
        [
            "dataset_id",
            "table_name",
            "cid",
            "name",
            "type",
            "notnull",
            "dflt_value",
            "pk",
        ]
    ]
)


# ---------------------------------------------------------------------------
# Display contents
# ---------------------------------------------------------------------------

print("\nPrecursor metadata and QA contents")
print("----------------------------------")

display(precursor_doc_contents)

Precursor metadata and QA table inventory
-----------------------------------------


,dataset_id,table_name,gpkg_data_type,row_count
0,aer_agreements,metadata_aer_agreements,attributes,25
1,aer_agreements,qa_aer_agreements,attributes,21
2,gbc_ne_atlas,metadata_gbc_ne_atlas,attributes,27
3,gbc_ne_atlas,qa_gbc_ne_atlas,attributes,21
4,gsc_atlantic,metadata_gsc_atlantic,attributes,28
5,gsc_atlantic,qa_gsc_atlantic,attributes,12
6,natcarb_doe,metadata_natcarb_doe,attributes,23
7,natcarb_doe,qa_natcarb_doe,attributes,11



Precursor metadata and QA schemas
---------------------------------


,dataset_id,table_name,cid,name,type,notnull,dflt_value,pk
0,aer_agreements,metadata_aer_agreements,0,key,TEXT,0,None,0
1,aer_agreements,metadata_aer_agreements,1,value,TEXT,0,None,0
2,aer_agreements,qa_aer_agreements,0,check,TEXT,0,None,0
3,aer_agreements,qa_aer_agreements,1,value,TEXT,0,None,0
4,gbc_ne_atlas,metadata_gbc_ne_atlas,0,key,TEXT,0,None,0
5,gbc_ne_atlas,metadata_gbc_ne_atlas,1,value,TEXT,0,None,0
6,gbc_ne_atlas,qa_gbc_ne_atlas,0,check,TEXT,0,None,0
7,gbc_ne_atlas,qa_gbc_ne_atlas,1,value,TEXT,0,None,0
8,gsc_atlantic,metadata_gsc_atlantic,0,key,TEXT,0,None,0
9,gsc_atlantic,metadata_gsc_atlantic,1,value,TEXT,0,None,0



Precursor metadata and QA contents
----------------------------------


,dataset_id,table_name,key,value,check,notes
0,aer_agreements,metadata_aer_agreements,title,Alberta Carbon Sequestration Agreements,NaN,NaN
1,aer_agreements,metadata_aer_agreements,who,"Andrew Vigars, CanCO2Re Activity 13",NaN,NaN
2,aer_agreements,metadata_aer_agreements,when,2026-09-13,NaN,NaN
3,aer_agreements,metadata_aer_agreements,submission_filename,20260913_13_AERCarbonSequestrationAgreements_A...,NaN,NaN
4,aer_agreements,metadata_aer_agreements,activity_code,13,NaN,NaN
...,...,...,...,...,...,...
163,natcarb_doe,qa_natcarb_doe,NaN,0,unassessed_with_capacity,Expected to be zero under documented NATCARB s...
164,natcarb_doe,qa_natcarb_doe,NaN,0,p10_gt_p50,Reported for source QA; values are not silentl...
165,natcarb_doe,qa_natcarb_doe,NaN,0,p50_gt_p90,Reported for source QA; values are not silentl...
166,natcarb_doe,qa_natcarb_doe,NaN,0,persisted_invalid_geometries,Persisted feature layers are reopened and requ...


In [9]:
# ---------------------------------------------------------------------------
# Compare precursor metadata keys and QA checks
# ---------------------------------------------------------------------------

metadata_rows = precursor_doc_contents[
    precursor_doc_contents["table_name"].str.startswith("metadata_")
].copy()

qa_rows = precursor_doc_contents[
    precursor_doc_contents["table_name"].str.startswith("qa_")
].copy()


# ---------------------------------------------------------------------------
# Metadata keys by dataset
# ---------------------------------------------------------------------------

metadata_key_matrix = (
    metadata_rows
    .dropna(subset=["key"])
    .assign(present=True)
    .pivot_table(
        index="key",
        columns="dataset_id",
        values="present",
        aggfunc="first",
        fill_value=False,
    )
    .astype(bool)
)

metadata_key_matrix["dataset_count"] = (
    metadata_key_matrix.sum(axis=1)
)

metadata_key_matrix = (
    metadata_key_matrix
    .sort_values(
        ["dataset_count", "key"],
        ascending=[False, True],
    )
)


# ---------------------------------------------------------------------------
# QA checks by dataset
# ---------------------------------------------------------------------------

qa_check_matrix = (
    qa_rows
    .dropna(subset=["check"])
    .assign(present=True)
    .pivot_table(
        index="check",
        columns="dataset_id",
        values="present",
        aggfunc="first",
        fill_value=False,
    )
    .astype(bool)
)

qa_check_matrix["dataset_count"] = (
    qa_check_matrix.sum(axis=1)
)

qa_check_matrix = (
    qa_check_matrix
    .sort_values(
        ["dataset_count", "check"],
        ascending=[False, True],
    )
)


# ---------------------------------------------------------------------------
# Common and dataset-specific fields
# ---------------------------------------------------------------------------

common_metadata_keys = metadata_key_matrix[
    metadata_key_matrix["dataset_count"] == len(SOURCE_GPKGS)
].index.tolist()

common_qa_checks = qa_check_matrix[
    qa_check_matrix["dataset_count"] == len(SOURCE_GPKGS)
].index.tolist()


print("Metadata key coverage")
print("---------------------")
display(metadata_key_matrix)


print("\nMetadata keys present in all precursor datasets")
print("-----------------------------------------------")

for key in common_metadata_keys:
    print(f"- {key}")


print("\nQA check coverage")
print("-----------------")
display(qa_check_matrix)


print("\nQA checks present in all precursor datasets")
print("--------------------------------------------")

for check in common_qa_checks:
    print(f"- {check}")

Metadata key coverage
---------------------


dataset_id,aer_agreements,gbc_ne_atlas,gsc_atlantic,natcarb_doe,dataset_count
key,,,,,
activity_code,True,True,True,True,4
assessment_type,True,True,True,True,4
capacity_data,True,True,True,True,4
creator_initials,True,True,True,True,4
data_class,True,True,True,True,4
silver_crs,True,True,True,True,4
source_organization,True,True,True,True,4
source_title,True,True,True,True,4
submission_filename,True,True,True,True,4



Metadata keys present in all precursor datasets
-----------------------------------------------
- activity_code
- assessment_type
- capacity_data
- creator_initials
- data_class
- silver_crs
- source_organization
- source_title
- submission_filename

QA check coverage
-----------------


dataset_id,aer_agreements,gbc_ne_atlas,gsc_atlantic,natcarb_doe,dataset_count
check,,,,,
silver_crs,True,True,True,True,4
feature_layers,False,False,True,True,2
persisted_invalid_geometries,False,False,True,True,2
total_spatial_features,False,False,True,True,2
agreement_empty_geometry_count,True,False,False,False,1
agreement_feature_count,True,False,False,False,1
agreement_invalid_geometry_count,True,False,False,False,1
agreement_null_geometry_count,True,False,False,False,1
aquifer_final_invalid_geometry_count,False,True,False,False,1



QA checks present in all precursor datasets
--------------------------------------------
- silver_crs


In [10]:
# ---------------------------------------------------------------------------
# Normalize precursor metadata keys to the established lowercase schema
# ---------------------------------------------------------------------------

METADATA_KEY_ALIASES = {
    "Who": "who",
    "What": "what",
    "When": "when",
    "Where": "where",
    "How": "how",
    "canco2re_activity": "activity_code",
}

metadata_normalized = metadata_rows.copy()

metadata_normalized["canonical_key"] = (
    metadata_normalized["key"]
    .replace(METADATA_KEY_ALIASES)
    .str.strip()
    .str.lower()
)


# ---------------------------------------------------------------------------
# Inspect normalized key coverage
# ---------------------------------------------------------------------------

metadata_normalized_matrix = (
    metadata_normalized
    .dropna(subset=["canonical_key"])
    .assign(present=True)
    .pivot_table(
        index="canonical_key",
        columns="dataset_id",
        values="present",
        aggfunc="first",
        fill_value=False,
    )
    .astype(bool)
)

metadata_normalized_matrix["dataset_count"] = (
    metadata_normalized_matrix.sum(axis=1)
)

metadata_normalized_matrix = (
    metadata_normalized_matrix
    .sort_values(
        ["dataset_count", "canonical_key"],
        ascending=[False, True],
    )
)


print("Normalized precursor metadata key coverage")
print("------------------------------------------")

display(metadata_normalized_matrix)

Normalized precursor metadata key coverage
------------------------------------------


dataset_id,aer_agreements,gbc_ne_atlas,gsc_atlantic,natcarb_doe,dataset_count
canonical_key,,,,,
activity_code,True,True,True,True,4
assessment_type,True,True,True,True,4
capacity_data,True,True,True,True,4
creator_initials,True,True,True,True,4
data_class,True,True,True,True,4
how,True,True,True,True,4
silver_crs,True,True,True,True,4
source_organization,True,True,True,True,4
source_title,True,True,True,True,4


In [11]:
# ---------------------------------------------------------------------------
# Compare normalized precursor metadata values
# ---------------------------------------------------------------------------

CORE_METADATA_KEYS = [
    "activity_code",
    "assessment_type",
    "capacity_data",
    "creator_initials",
    "data_class",
    "how",
    "silver_crs",
    "source_organization",
    "source_title",
    "submission_filename",
    "what",
    "when",
    "where",
    "who",
]

EXTENDED_METADATA_KEYS = [
    "dataset_id",
    "injectivity_status",
    "source_url",
    "source_year",
    "source_publication",
    "source_doi",
    "licence_name",
    "licence_url",
    "attribution_text",
    "interpretation_note",
    "use_limitations",
    "keywords",
]

metadata_value_matrix = (
    metadata_normalized
    .dropna(subset=["canonical_key"])
    .loc[
        metadata_normalized["canonical_key"].isin(
            CORE_METADATA_KEYS + EXTENDED_METADATA_KEYS
        ),
        [
            "dataset_id",
            "canonical_key",
            "value",
        ],
    ]
    .pivot_table(
        index="canonical_key",
        columns="dataset_id",
        values="value",
        aggfunc="first",
    )
    .reindex(
        CORE_METADATA_KEYS + EXTENDED_METADATA_KEYS
    )
)


print("Normalized precursor metadata values")
print("------------------------------------")

display(metadata_value_matrix)

Normalized precursor metadata values
------------------------------------


dataset_id,aer_agreements,gbc_ne_atlas,gsc_atlantic,natcarb_doe
canonical_key,,,,
activity_code,13,13,13,13
assessment_type,carbon_sequestration_agreement,quantitative_geological_storage_screening,qualitative_chance_of_success,quantitative_geological_storage_resource
capacity_data,False,True,False,True
creator_initials,AV,AV,AV,AV
data_class,regulatory_tenure,geological_storage_capacity,geological_prospectivity,geological_storage_capacity
how,Derived from the Alberta Energy Regulator Carb...,Pool classifications and logical pool attribut...,Derived from 15 Geological Survey of Canada Op...,Derived from the NATCARB v1502 File Geodatabas...
silver_crs,EPSG:3978,EPSG:3978,EPSG:3978,EPSG:3978
source_organization,Alberta Energy Regulator (AER),Geoscience BC,"Natural Resources Canada, Geological Survey of...","U.S. Department of Energy, National Energy Tec..."
source_title,Carbon Sequestration Agreements,Northeast BC Geological Carbon Capture and Sto...,Preliminary assessment of geological carbon-st...,NATCARB All Data v1502


In [12]:
# ---------------------------------------------------------------------------
# Compare precursor QA checks and values
# ---------------------------------------------------------------------------

qa_values = (
    qa_rows
    .dropna(subset=["check"])
    .loc[
        :,
        [
            "dataset_id",
            "check",
            "value",
            "notes",
        ],
    ]
    .copy()
)


# ---------------------------------------------------------------------------
# QA check coverage across precursor datasets
# ---------------------------------------------------------------------------

qa_check_matrix = (
    qa_values
    .assign(present=True)
    .pivot_table(
        index="check",
        columns="dataset_id",
        values="present",
        aggfunc="first",
        fill_value=False,
    )
    .astype(bool)
)

qa_check_matrix["dataset_count"] = (
    qa_check_matrix.sum(axis=1)
)

qa_check_matrix = (
    qa_check_matrix
    .sort_values(
        ["dataset_count", "check"],
        ascending=[False, True],
    )
)


# ---------------------------------------------------------------------------
# QA values side by side
# ---------------------------------------------------------------------------

qa_value_matrix = (
    qa_values
    .pivot_table(
        index="check",
        columns="dataset_id",
        values="value",
        aggfunc="first",
    )
    .reindex(qa_check_matrix.index)
)


# ---------------------------------------------------------------------------
# QA notes side by side
# ---------------------------------------------------------------------------

qa_notes_matrix = (
    qa_values
    .pivot_table(
        index="check",
        columns="dataset_id",
        values="notes",
        aggfunc="first",
    )
    .reindex(qa_check_matrix.index)
)


# ---------------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------------

print("Precursor QA check coverage")
print("---------------------------")

display(qa_check_matrix)


print("\nPrecursor QA values")
print("-------------------")

display(qa_value_matrix)


print("\nPrecursor QA notes")
print("------------------")

display(qa_notes_matrix)

Precursor QA check coverage
---------------------------


dataset_id,aer_agreements,gbc_ne_atlas,gsc_atlantic,natcarb_doe,dataset_count
check,,,,,
silver_crs,True,True,True,True,4
feature_layers,False,False,True,True,2
persisted_invalid_geometries,False,False,True,True,2
total_spatial_features,False,False,True,True,2
agreement_empty_geometry_count,True,False,False,False,1
agreement_feature_count,True,False,False,False,1
agreement_invalid_geometry_count,True,False,False,False,1
agreement_null_geometry_count,True,False,False,False,1
aquifer_final_invalid_geometry_count,False,True,False,False,1



Precursor QA values
-------------------


dataset_id,aer_agreements,gbc_ne_atlas,gsc_atlantic,natcarb_doe
check,,,,
silver_crs,EPSG:3978,EPSG:3978,EPSG:3978,EPSG:3978
feature_layers,NaN,NaN,1,5
persisted_invalid_geometries,NaN,NaN,0,0
total_spatial_features,NaN,NaN,4711,274102
agreement_empty_geometry_count,0,NaN,NaN,NaN
agreement_feature_count,43,NaN,NaN,NaN
agreement_invalid_geometry_count,0,NaN,NaN,NaN
agreement_null_geometry_count,0,NaN,NaN,NaN
aquifer_final_invalid_geometry_count,NaN,0,NaN,NaN



Precursor QA notes
------------------


dataset_id,gsc_atlantic,natcarb_doe
check,,
silver_crs,Canonical repository Silver CRS.,All persisted spatial layers use the CanCO2Re ...
feature_layers,One harmonized geological prospectivity featur...,Five geological storage feature layers are exp...
persisted_invalid_geometries,Persisted Silver geometries are required to be...,Persisted feature layers are reopened and requ...
total_spatial_features,Total source features preserved in the Silver ...,Total harmonized spatial features.
agreement_empty_geometry_count,NaN,NaN
agreement_feature_count,NaN,NaN
agreement_invalid_geometry_count,NaN,NaN
agreement_null_geometry_count,NaN,NaN
aquifer_final_invalid_geometry_count,NaN,NaN


In [13]:
# ---------------------------------------------------------------------------
# Read precursor metadata and QA directly from source GeoPackages
# ---------------------------------------------------------------------------

SOURCE_DOCUMENTATION_TABLES = {
    "aer_agreements": {
        "metadata": "metadata_aer_agreements",
        "qa": "qa_aer_agreements",
    },
    "gbc_ne_atlas": {
        "metadata": "metadata_gbc_ne_atlas",
        "qa": "qa_gbc_ne_atlas",
    },
    "gsc_atlantic": {
        "metadata": "metadata_gsc_atlantic",
        "qa": "qa_gsc_atlantic",
    },
    "natcarb_doe": {
        "metadata": "metadata_natcarb_doe",
        "qa": "qa_natcarb_doe",
    },
}


source_metadata_frames = []
source_qa_frames = []


for dataset_id, gpkg_path in SOURCE_GPKGS.items():

    table_names = SOURCE_DOCUMENTATION_TABLES[dataset_id]

    conn = sqlite3.connect(gpkg_path)

    try:
        # -------------------------------------------------------------------
        # Metadata
        # -------------------------------------------------------------------

        metadata = pd.read_sql_query(
            f"""
            SELECT
                key,
                value
            FROM "{table_names['metadata']}";
            """,
            conn,
        )

        metadata.insert(
            0,
            "dataset_id",
            dataset_id,
        )

        source_metadata_frames.append(metadata)

        # -------------------------------------------------------------------
        # QA
        # -------------------------------------------------------------------

        qa = pd.read_sql_query(
            f"""
            SELECT *
            FROM "{table_names['qa']}";
            """,
            conn,
        )

        if "notes" not in qa.columns:
            qa["notes"] = None

        qa = qa[
            [
                "check",
                "value",
                "notes",
            ]
        ]

        qa.insert(
            0,
            "dataset_id",
            dataset_id,
        )

        source_qa_frames.append(qa)

    finally:
        conn.close()


# ---------------------------------------------------------------------------
# Combine precursor documentation
# ---------------------------------------------------------------------------

source_metadata = (
    pd.concat(
        source_metadata_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "dataset_id",
            "key",
        ]
    )
    .reset_index(drop=True)
)


source_qa = (
    pd.concat(
        source_qa_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "dataset_id",
            "check",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Review
# ---------------------------------------------------------------------------

print("Unified precursor metadata lineage")
print("----------------------------------")

display(source_metadata)


print("\nUnified precursor QA lineage")
print("----------------------------")

display(source_qa)

Unified precursor metadata lineage
----------------------------------


,dataset_id,key,value
0,aer_agreements,activity_code,13
1,aer_agreements,assessment_type,carbon_sequestration_agreement
2,aer_agreements,bronze_layer_count,1
3,aer_agreements,capacity_data,False
4,aer_agreements,creator_initials,AV
...,...,...,...
98,natcarb_doe,source_organization,"U.S. Department of Energy, National Energy Tec..."
99,natcarb_doe,source_publication_date,2015-05-14
100,natcarb_doe,source_title,NATCARB All Data v1502
101,natcarb_doe,source_version,v1502



Unified precursor QA lineage
----------------------------


,dataset_id,check,value,notes
0,aer_agreements,agreement_empty_geometry_count,0,None
1,aer_agreements,agreement_feature_count,43,None
2,aer_agreements,agreement_invalid_geometry_count,0,None
3,aer_agreements,agreement_null_geometry_count,0,None
4,aer_agreements,median_source_to_silver_area_ratio,1.03139455232659,None
...,...,...,...,...
60,natcarb_doe,p50_gt_p90,0,Reported for source QA; values are not silentl...
61,natcarb_doe,persisted_invalid_geometries,0,Persisted feature layers are reopened and requ...
62,natcarb_doe,silver_crs,EPSG:3978,All persisted spatial layers use the CanCO2Re ...
63,natcarb_doe,total_spatial_features,274102,Total harmonized spatial features.


In [14]:
# ---------------------------------------------------------------------------
# Build unified dataset-level metadata in memory
# ---------------------------------------------------------------------------

unified_metadata = pd.DataFrame(
    [
        (
            "title",
            "Canada Geological CO2 Storage Unified Database",
        ),
        (
            "dataset_id",
            "canada_geological_storage_unified",
        ),
        (
            "data_type",
            "CanadaGeologicalStorageUnified",
        ),
        (
            "who",
            "Andrew Vigars, CanCO2Re Activity 13",
        ),
        (
            "what",
            (
                "Unified Canadian geological CO2 storage database integrating "
                "harmonized geological storage capacity, geological prospectivity, "
                "and regulatory tenure datasets from multiple precursor Silver "
                "GeoPackages."
            ),
        ),
        (
            "when",
            "2026-09-16",
        ),
        (
            "where",
            (
                "Canada. Spatial layers use NAD83 / Canada Atlas Lambert, "
                "EPSG:3978."
            ),
        ),
        (
            "how",
            (
                "Constructed by harmonizing and integrating precursor Silver "
                "GeoPackages from Alberta Energy Regulator carbon sequestration "
                "agreements, the Northeast BC Geological Carbon Capture and Storage "
                "Atlas, Geological Survey of Canada Atlantic Chance of Success "
                "mapping, and the Canadian subset of DOE/NETL NATCARB v1502."
            ),
        ),
        (
            "activity_code",
            "13",
        ),
        (
            "creator_name",
            "Andrew Vigars",
        ),
        (
            "creator_initials",
            "AV",
        ),
        (
            "submission_filename",
            GPKG_PATH.name,
        ),
        (
            "dataset_role",
            (
                "Unified Silver-to-integration-layer database for national-scale "
                "geological CO2 storage screening, comparison, spatial analysis, "
                "and downstream energy-system and CCUS modelling."
            ),
        ),
        (
            "package_purpose",
            (
                "Provides a common national schema for geological storage units, "
                "spatial representations, storage assessments, and regulatory "
                "administrative features while preserving source provenance and "
                "source-specific interpretation."
            ),
        ),
        (
            "silver_crs",
            "EPSG:3978",
        ),
        (
            "silver_format",
            "GeoPackage",
        ),
        (
            "assessment_type",
            "mixed_geological_storage_assessment",
        ),
        (
            "data_class",
            "integrated_geological_storage",
        ),
        (
            "capacity_data",
            "mixed",
        ),
        (
            "capacity_status",
            "source_dependent",
        ),
        (
            "injectivity_status",
            "source_dependent",
        ),
        (
            "interpretation_note",
            (
                "The unified database integrates datasets with different scientific "
                "and regulatory meanings. Geological capacity, geological "
                "prospectivity, and regulatory tenure records must not be treated "
                "as interchangeable. Source_dataset and assessment_type fields "
                "should be used when interpreting individual records."
            ),
        ),
        (
            "processing_summary",
            (
                "Precursor Silver datasets were mapped into a canonical schema "
                "consisting of storage_units, storage_features, "
                "storage_assessments, and administrative_features. Stable "
                "source-derived identifiers and source_dataset provenance were "
                "preserved. Logical storage units were separated from spatial "
                "representations, assessment scope was retained explicitly, and "
                "regulatory tenure polygons were kept separate from geological "
                "storage objects."
            ),
        ),
        (
            "use_limitations",
            (
                "This database is intended for regional and national screening, "
                "comparative analysis, and model input preparation. It does not "
                "establish project-ready storage capacity, demonstrated injectivity, "
                "permitted injection capacity, legal storage rights, or site-specific "
                "geological suitability. Source-specific limitations remain "
                "authoritative and should be consulted through source_metadata."
            ),
        ),
        (
            "keywords",
            (
                "carbon storage; CO2 storage; CCUS; geological storage; "
                "saline aquifer; depleted reservoir; geological prospectivity; "
                "carbon sequestration agreement; Canada"
            ),
        ),
    ],
    columns=[
        "key",
        "value",
    ],
)


print("Unified dataset-level metadata")
print("------------------------------")

display(unified_metadata)

Unified dataset-level metadata
------------------------------


,key,value
0,title,Canada Geological CO2 Storage Unified Database
1,dataset_id,canada_geological_storage_unified
2,data_type,CanadaGeologicalStorageUnified
3,who,"Andrew Vigars, CanCO2Re Activity 13"
4,what,Unified Canadian geological CO2 storage databa...
5,when,2026-09-16
6,where,Canada. Spatial layers use NAD83 / Canada Atla...
7,how,Constructed by harmonizing and integrating pre...
8,activity_code,13
9,creator_name,Andrew Vigars


In [15]:
# ---------------------------------------------------------------------------
# Finalize unified dataset-level metadata values
# ---------------------------------------------------------------------------

UNIFIED_DATA_TYPE = "CanadaGeologicalStorageUnified"
UNIFIED_DATE = "2026-09-17"

UNIFIED_SUBMISSION_FILENAME = (
    "20260917_13_"
    f"{UNIFIED_DATA_TYPE}_"
    "AV.gpkg"
)


unified_metadata = pd.DataFrame(
    [
        (
            "title",
            "Canada Geological CO2 Storage Unified Database",
        ),
        (
            "dataset_id",
            "canada_geological_storage_unified",
        ),
        (
            "data_type",
            UNIFIED_DATA_TYPE,
        ),
        (
            "who",
            "Andrew Vigars, CanCO2Re Activity 13",
        ),
        (
            "what",
            (
                "Unified Canadian geological CO2 storage database integrating "
                "harmonized geological storage capacity, geological prospectivity, "
                "and regulatory tenure datasets from multiple precursor Silver "
                "GeoPackages."
            ),
        ),
        (
            "when",
            UNIFIED_DATE,
        ),
        (
            "where",
            (
                "Canada. Spatial layers use NAD83 / Canada Atlas Lambert, "
                "EPSG:3978."
            ),
        ),
        (
            "how",
            (
                "Constructed by harmonizing and integrating precursor Silver "
                "GeoPackages from Alberta Energy Regulator carbon sequestration "
                "agreements, the Northeast BC Geological Carbon Capture and Storage "
                "Atlas, Geological Survey of Canada Atlantic Chance of Success "
                "mapping, and the Canadian subset of DOE/NETL NATCARB v1502."
            ),
        ),
        (
            "activity_code",
            "13",
        ),
        (
            "creator_name",
            "Andrew Vigars",
        ),
        (
            "creator_initials",
            "AV",
        ),
        (
            "submission_filename",
            UNIFIED_SUBMISSION_FILENAME,
        ),
        (
            "dataset_role",
            (
                "Unified integration-layer database for national-scale geological "
                "CO2 storage screening, comparison, spatial analysis, and "
                "downstream CCUS modelling."
            ),
        ),
        (
            "package_purpose",
            (
                "Provides a common national schema for geological storage units, "
                "spatial representations, storage assessments, and regulatory "
                "administrative features while preserving source provenance and "
                "source-specific interpretation."
            ),
        ),
        (
            "silver_crs",
            "EPSG:3978",
        ),
        (
            "silver_format",
            "GeoPackage",
        ),
        (
            "assessment_type",
            "mixed_geological_storage_assessment",
        ),
        (
            "data_class",
            "integrated_geological_storage",
        ),
        (
            "capacity_data",
            "mixed",
        ),
        (
            "capacity_status",
            "source_dependent",
        ),
        (
            "injectivity_status",
            "source_dependent",
        ),
        (
            "interpretation_note",
            (
                "The unified database integrates datasets with different scientific "
                "and regulatory meanings. Geological capacity, geological "
                "prospectivity, and regulatory tenure records must not be treated "
                "as interchangeable. Source_dataset and assessment_type fields "
                "should be used when interpreting individual records."
            ),
        ),
        (
            "processing_summary",
            (
                "Precursor Silver datasets were mapped into a canonical schema "
                "consisting of storage_units, storage_features, "
                "storage_assessments, and administrative_features. Stable "
                "source-derived identifiers and source_dataset provenance were "
                "preserved. Logical storage units were separated from spatial "
                "representations, assessment scope was retained explicitly, and "
                "regulatory tenure polygons were kept separate from geological "
                "storage objects."
            ),
        ),
        (
            "use_limitations",
            (
                "This database is intended for regional and national screening, "
                "comparative analysis, and model input preparation. It does not "
                "establish project-ready storage capacity, demonstrated injectivity, "
                "permitted injection capacity, legal storage rights, or site-specific "
                "geological suitability. Source-specific limitations remain "
                "authoritative and should be consulted through source_metadata."
            ),
        ),
        (
            "keywords",
            (
                "carbon storage; CO2 storage; CCUS; geological storage; "
                "saline aquifer; depleted reservoir; geological prospectivity; "
                "carbon sequestration agreement; Canada"
            ),
        ),
    ],
    columns=[
        "key",
        "value",
    ],
)


print("Unified dataset-level metadata")
print("------------------------------")

display(unified_metadata)

Unified dataset-level metadata
------------------------------


,key,value
0,title,Canada Geological CO2 Storage Unified Database
1,dataset_id,canada_geological_storage_unified
2,data_type,CanadaGeologicalStorageUnified
3,who,"Andrew Vigars, CanCO2Re Activity 13"
4,what,Unified Canadian geological CO2 storage databa...
5,when,2026-09-17
6,where,Canada. Spatial layers use NAD83 / Canada Atla...
7,how,Constructed by harmonizing and integrating pre...
8,activity_code,13
9,creator_name,Andrew Vigars


In [16]:
# ---------------------------------------------------------------------------
# Validate unified dataset-level metadata contract
# ---------------------------------------------------------------------------

REQUIRED_METADATA_KEYS = {
    "title",
    "dataset_id",
    "data_type",
    "who",
    "what",
    "when",
    "where",
    "how",
    "activity_code",
    "creator_name",
    "creator_initials",
    "submission_filename",
    "dataset_role",
    "package_purpose",
    "silver_crs",
    "silver_format",
    "assessment_type",
    "data_class",
    "capacity_data",
    "capacity_status",
    "injectivity_status",
    "interpretation_note",
    "processing_summary",
    "use_limitations",
    "keywords",
}


# ---------------------------------------------------------------------------
# Basic key/value checks
# ---------------------------------------------------------------------------

if unified_metadata["key"].isna().any():
    raise ValueError("Unified metadata contains null keys.")

if unified_metadata["key"].duplicated().any():
    duplicate_keys = (
        unified_metadata.loc[
            unified_metadata["key"].duplicated(keep=False),
            "key",
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Unified metadata contains duplicate keys: {duplicate_keys}"
    )

blank_keys = (
    unified_metadata["key"]
    .astype(str)
    .str.strip()
    .eq("")
)

if blank_keys.any():
    raise ValueError("Unified metadata contains blank keys.")


# ---------------------------------------------------------------------------
# Required-key coverage
# ---------------------------------------------------------------------------

metadata_keys = set(unified_metadata["key"])

missing_keys = REQUIRED_METADATA_KEYS - metadata_keys
unexpected_keys = metadata_keys - REQUIRED_METADATA_KEYS

if missing_keys:
    raise ValueError(
        "Unified metadata is missing required keys: "
        f"{sorted(missing_keys)}"
    )


# ---------------------------------------------------------------------------
# Convert to lookup for semantic validation
# ---------------------------------------------------------------------------

metadata_lookup = dict(
    zip(
        unified_metadata["key"],
        unified_metadata["value"],
    )
)


# ---------------------------------------------------------------------------
# Validate known package-level values
# ---------------------------------------------------------------------------

EXPECTED_VALUES = {
    "dataset_id": "canada_geological_storage_unified",
    "data_type": "CanadaGeologicalStorageUnified",
    "activity_code": "13",
    "creator_initials": "AV",
    "silver_crs": "EPSG:3978",
    "silver_format": "GeoPackage",
}

for key, expected_value in EXPECTED_VALUES.items():

    actual_value = str(metadata_lookup[key])

    if actual_value != expected_value:
        raise ValueError(
            f"{key!r} expected {expected_value!r}, "
            f"found {actual_value!r}."
        )


# ---------------------------------------------------------------------------
# Validate submission filename convention
# ---------------------------------------------------------------------------

expected_submission_filename = (
    f"{metadata_lookup['when'].replace('-', '')}_"
    f"{metadata_lookup['activity_code']}_"
    f"{metadata_lookup['data_type']}_"
    f"{metadata_lookup['creator_initials']}.gpkg"
)

if metadata_lookup["submission_filename"] != expected_submission_filename:
    raise ValueError(
        "submission_filename does not match the established "
        "CanCO2Re naming convention.\n"
        f"Expected: {expected_submission_filename}\n"
        f"Found:    {metadata_lookup['submission_filename']}"
    )


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("Unified metadata validation")
print("---------------------------")
print(f"Rows:                 {len(unified_metadata)}")
print(f"Unique keys:          {unified_metadata['key'].nunique()}")
print(f"Required keys:        {len(REQUIRED_METADATA_KEYS)}")
print(f"Missing keys:         {len(missing_keys)}")
print(f"Unexpected keys:      {len(unexpected_keys)}")
print(f"Submission filename:  {metadata_lookup['submission_filename']}")
print(f"CRS:                  {metadata_lookup['silver_crs']}")
print("\nMetadata contract passed.")

Unified metadata validation
---------------------------
Rows:                 25
Unique keys:          25
Required keys:        25
Missing keys:         0
Unexpected keys:      0
Submission filename:  20260917_13_CanadaGeologicalStorageUnified_AV.gpkg
CRS:                  EPSG:3978

Metadata contract passed.


In [17]:
# ---------------------------------------------------------------------------
# Build unified dataset-level QA in memory
# ---------------------------------------------------------------------------

unified_qa_rows = []

conn = sqlite3.connect(GPKG_PATH)

try:
    # -----------------------------------------------------------------------
    # Canonical table counts
    # -----------------------------------------------------------------------

    for table_name in CANONICAL_TABLES:
        row_count = conn.execute(
            f'SELECT COUNT(*) FROM "{table_name}";'
        ).fetchone()[0]

        unified_qa_rows.append(
            {
                "check": f"{table_name}_count",
                "value": row_count,
                "notes": f"Persisted row count for {table_name}.",
            }
        )

    # -----------------------------------------------------------------------
    # Identifier integrity
    # -----------------------------------------------------------------------

    ID_FIELDS = {
        "storage_units": "storage_unit_id",
        "storage_features": "storage_feature_id",
        "storage_assessments": "storage_assessment_id",
        "administrative_features": "administrative_feature_id",
    }

    for table_name, id_field in ID_FIELDS.items():

        missing_count = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM "{table_name}"
            WHERE "{id_field}" IS NULL
               OR TRIM(CAST("{id_field}" AS TEXT)) = '';
            """
        ).fetchone()[0]

        duplicate_count = conn.execute(
            f"""
            SELECT COUNT(*) - COUNT(DISTINCT "{id_field}")
            FROM "{table_name}"
            WHERE "{id_field}" IS NOT NULL;
            """
        ).fetchone()[0]

        unified_qa_rows.extend(
            [
                {
                    "check": f"{table_name}_missing_id_count",
                    "value": missing_count,
                    "notes": f"Null or blank values in {id_field}.",
                },
                {
                    "check": f"{table_name}_duplicate_id_count",
                    "value": duplicate_count,
                    "notes": f"Duplicate non-null values in {id_field}.",
                },
            ]
        )

    # -----------------------------------------------------------------------
    # Logical relationship integrity
    # -----------------------------------------------------------------------

    relationship_checks = {
        "storage_features_orphan_unit_count": """
            SELECT COUNT(*)
            FROM storage_features AS f
            LEFT JOIN storage_units AS u
                ON f.storage_unit_id = u.storage_unit_id
            WHERE f.storage_unit_id IS NOT NULL
              AND u.storage_unit_id IS NULL;
        """,
        "storage_assessments_orphan_unit_count": """
            SELECT COUNT(*)
            FROM storage_assessments AS a
            LEFT JOIN storage_units AS u
                ON a.storage_unit_id = u.storage_unit_id
            WHERE a.storage_unit_id IS NOT NULL
              AND u.storage_unit_id IS NULL;
        """,
        "storage_assessments_orphan_feature_count": """
            SELECT COUNT(*)
            FROM storage_assessments AS a
            LEFT JOIN storage_features AS f
                ON a.storage_feature_id = f.storage_feature_id
            WHERE a.storage_feature_id IS NOT NULL
              AND f.storage_feature_id IS NULL;
        """,
        "administrative_features_orphan_parent_count": """
            SELECT COUNT(*)
            FROM administrative_features AS child
            LEFT JOIN administrative_features AS parent
                ON child.parent_administrative_feature_id
                 = parent.administrative_feature_id
            WHERE child.parent_administrative_feature_id IS NOT NULL
              AND parent.administrative_feature_id IS NULL;
        """,
    }

    for check_name, sql in relationship_checks.items():
        value = conn.execute(sql).fetchone()[0]

        unified_qa_rows.append(
            {
                "check": check_name,
                "value": value,
                "notes": "Logical relationship integrity check.",
            }
        )

    # -----------------------------------------------------------------------
    # Spatial registration and CRS
    # -----------------------------------------------------------------------

    geometry_columns = pd.read_sql_query(
        """
        SELECT
            table_name,
            column_name,
            geometry_type_name,
            srs_id
        FROM gpkg_geometry_columns
        ORDER BY table_name;
        """,
        conn,
    )

    spatial_tables = geometry_columns["table_name"].tolist()
    spatial_crs = sorted(
        geometry_columns["srs_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    unified_qa_rows.append(
        {
            "check": "feature_layers",
            "value": len(spatial_tables),
            "notes": (
                "Number of registered spatial feature tables in "
                "gpkg_geometry_columns."
            ),
        }
    )

    unified_qa_rows.append(
        {
            "check": "silver_crs",
            "value": (
                f"EPSG:{spatial_crs[0]}"
                if len(spatial_crs) == 1
                else str(spatial_crs)
            ),
            "notes": (
                "CRS registered for persisted spatial feature tables."
            ),
        }
    )

    # -----------------------------------------------------------------------
    # Geometry null checks
    # -----------------------------------------------------------------------

    for table_name in spatial_tables:

        null_geometry_count = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM "{table_name}"
            WHERE geom IS NULL;
            """
        ).fetchone()[0]

        unified_qa_rows.append(
            {
                "check": f"{table_name}_null_geometry_count",
                "value": null_geometry_count,
                "notes": (
                    f"Persisted null geometry count for {table_name}."
                ),
            }
        )

finally:
    conn.close()


# ---------------------------------------------------------------------------
# Build QA table
# ---------------------------------------------------------------------------

unified_qa = (
    pd.DataFrame(unified_qa_rows)
    .sort_values("check")
    .reset_index(drop=True)
)


print("Unified dataset-level QA")
print("------------------------")

display(unified_qa)

Unified dataset-level QA
------------------------


,check,value,notes
0,administrative_features_count,88,Persisted row count for administrative_features.
1,administrative_features_duplicate_id_count,0,Duplicate non-null values in administrative_fe...
2,administrative_features_missing_id_count,0,Null or blank values in administrative_feature...
3,administrative_features_null_geometry_count,0,Persisted null geometry count for administrati...
4,administrative_features_orphan_parent_count,0,Logical relationship integrity check.
5,feature_layers,2,Number of registered spatial feature tables in...
6,silver_crs,EPSG:3978,CRS registered for persisted spatial feature t...
7,storage_assessments_count,35245,Persisted row count for storage_assessments.
8,storage_assessments_duplicate_id_count,0,Duplicate non-null values in storage_assessmen...
9,storage_assessments_missing_id_count,0,Null or blank values in storage_assessment_id.


In [18]:
# ---------------------------------------------------------------------------
# Add persisted geometry-validity QA checks
# ---------------------------------------------------------------------------

geometry_qa_rows = []

for table_name in [
    "storage_features",
    "administrative_features",
]:
    gdf = gpd.read_file(
        GPKG_PATH,
        layer=table_name,
    )

    invalid_geometry_count = int(
        (
            gdf.geometry.notna()
            & ~gdf.geometry.is_valid
        ).sum()
    )

    geometry_qa_rows.append(
        {
            "check": f"{table_name}_invalid_geometry_count",
            "value": invalid_geometry_count,
            "notes": (
                f"Persisted invalid geometry count for {table_name}."
            ),
        }
    )


unified_qa = (
    pd.concat(
        [
            unified_qa,
            pd.DataFrame(geometry_qa_rows),
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset="check",
        keep="last",
    )
    .sort_values("check")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Validate unified QA contract
# ---------------------------------------------------------------------------

REQUIRED_QA_CHECKS = {
    "administrative_features_count",
    "administrative_features_duplicate_id_count",
    "administrative_features_missing_id_count",
    "administrative_features_null_geometry_count",
    "administrative_features_invalid_geometry_count",
    "administrative_features_orphan_parent_count",
    "feature_layers",
    "silver_crs",
    "storage_assessments_count",
    "storage_assessments_duplicate_id_count",
    "storage_assessments_missing_id_count",
    "storage_assessments_orphan_feature_count",
    "storage_assessments_orphan_unit_count",
    "storage_features_count",
    "storage_features_duplicate_id_count",
    "storage_features_missing_id_count",
    "storage_features_null_geometry_count",
    "storage_features_invalid_geometry_count",
    "storage_features_orphan_unit_count",
    "storage_units_count",
    "storage_units_duplicate_id_count",
    "storage_units_missing_id_count",
}


# ---------------------------------------------------------------------------
# Structural validation
# ---------------------------------------------------------------------------

if unified_qa["check"].isna().any():
    raise ValueError("Unified QA contains null check names.")

if unified_qa["check"].duplicated().any():
    duplicate_checks = (
        unified_qa.loc[
            unified_qa["check"].duplicated(keep=False),
            "check",
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Unified QA contains duplicate checks: {duplicate_checks}"
    )


qa_lookup = dict(
    zip(
        unified_qa["check"],
        unified_qa["value"],
    )
)

missing_checks = (
    REQUIRED_QA_CHECKS
    - set(qa_lookup)
)

if missing_checks:
    raise ValueError(
        "Unified QA is missing required checks: "
        f"{sorted(missing_checks)}"
    )


# ---------------------------------------------------------------------------
# Zero-tolerance integrity checks
# ---------------------------------------------------------------------------

ZERO_EXPECTED_CHECKS = {
    "administrative_features_duplicate_id_count",
    "administrative_features_missing_id_count",
    "administrative_features_null_geometry_count",
    "administrative_features_invalid_geometry_count",
    "administrative_features_orphan_parent_count",
    "storage_assessments_duplicate_id_count",
    "storage_assessments_missing_id_count",
    "storage_assessments_orphan_feature_count",
    "storage_assessments_orphan_unit_count",
    "storage_features_duplicate_id_count",
    "storage_features_missing_id_count",
    "storage_features_null_geometry_count",
    "storage_features_invalid_geometry_count",
    "storage_features_orphan_unit_count",
    "storage_units_duplicate_id_count",
    "storage_units_missing_id_count",
}

failed_zero_checks = {
    check: qa_lookup[check]
    for check in ZERO_EXPECTED_CHECKS
    if int(qa_lookup[check]) != 0
}

if failed_zero_checks:
    raise ValueError(
        "Unified QA integrity checks failed: "
        f"{failed_zero_checks}"
    )


# ---------------------------------------------------------------------------
# Package-level expectations
# ---------------------------------------------------------------------------

if qa_lookup["silver_crs"] != "EPSG:3978":
    raise ValueError(
        f"Expected EPSG:3978, found {qa_lookup['silver_crs']}."
    )

if int(qa_lookup["feature_layers"]) != 2:
    raise ValueError(
        "Expected two registered spatial feature layers, "
        f"found {qa_lookup['feature_layers']}."
    )


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("Unified QA validation")
print("---------------------")
print(f"Checks:               {len(unified_qa)}")
print(f"Required checks:      {len(REQUIRED_QA_CHECKS)}")
print(f"Missing checks:       {len(missing_checks)}")
print(f"Failed zero checks:   {len(failed_zero_checks)}")
print(f"CRS:                  {qa_lookup['silver_crs']}")
print(f"Feature layers:       {qa_lookup['feature_layers']}")
print("\nUnified QA contract passed.")

display(unified_qa)

Unified QA validation
---------------------
Checks:               22
Required checks:      22
Missing checks:       0
Failed zero checks:   0
CRS:                  EPSG:3978
Feature layers:       2

Unified QA contract passed.


,check,value,notes
0,administrative_features_count,88,Persisted row count for administrative_features.
1,administrative_features_duplicate_id_count,0,Duplicate non-null values in administrative_fe...
2,administrative_features_invalid_geometry_count,0,Persisted invalid geometry count for administr...
3,administrative_features_missing_id_count,0,Null or blank values in administrative_feature...
4,administrative_features_null_geometry_count,0,Persisted null geometry count for administrati...
5,administrative_features_orphan_parent_count,0,Logical relationship integrity check.
6,feature_layers,2,Number of registered spatial feature tables in...
7,silver_crs,EPSG:3978,CRS registered for persisted spatial feature t...
8,storage_assessments_count,35245,Persisted row count for storage_assessments.
9,storage_assessments_duplicate_id_count,0,Duplicate non-null values in storage_assessmen...


In [19]:
# ---------------------------------------------------------------------------
# Define proposed final GeoPackage content contract
# ---------------------------------------------------------------------------

UNIFIED_DATASET_ID = "canada_geological_storage_unified"

UNIFIED_METADATA_TABLE = (
    f"metadata_{UNIFIED_DATASET_ID}"
)

UNIFIED_QA_TABLE = (
    f"qa_{UNIFIED_DATASET_ID}"
)


proposed_contents = pd.DataFrame(
    [
        {
            "table_name": "storage_units",
            "role": "canonical_domain",
            "spatial": False,
            "proposed_gpkg_type": None,
            "status": "review",
            "description": (
                "Logical geological storage units independent of "
                "spatial representation."
            ),
        },
        {
            "table_name": "storage_features",
            "role": "canonical_domain",
            "spatial": True,
            "proposed_gpkg_type": "features",
            "status": "retain",
            "description": (
                "Spatial representations of geological storage units."
            ),
        },
        {
            "table_name": "storage_assessments",
            "role": "canonical_domain",
            "spatial": False,
            "proposed_gpkg_type": None,
            "status": "review",
            "description": (
                "Quantitative and qualitative assessments attached "
                "to storage units or spatial features."
            ),
        },
        {
            "table_name": "administrative_features",
            "role": "canonical_domain",
            "spatial": True,
            "proposed_gpkg_type": "features",
            "status": "retain",
            "description": (
                "Regulatory and administrative carbon sequestration "
                "features kept separate from geological storage objects."
            ),
        },
        {
            "table_name": UNIFIED_METADATA_TABLE,
            "role": "dataset_metadata",
            "spatial": False,
            "proposed_gpkg_type": "attributes",
            "status": "add",
            "description": (
                "Authoritative dataset-level provenance, interpretation, "
                "classification, processing, and submission metadata."
            ),
        },
        {
            "table_name": UNIFIED_QA_TABLE,
            "role": "dataset_qa",
            "spatial": False,
            "proposed_gpkg_type": "attributes",
            "status": "add",
            "description": (
                "Authoritative quality-assurance summary for the "
                "unified persisted artifact."
            ),
        },
        {
            "table_name": "source_metadata",
            "role": "source_lineage",
            "spatial": False,
            "proposed_gpkg_type": "attributes",
            "status": "add",
            "description": (
                "Merged authoritative metadata lineage copied from "
                "the four precursor Silver GeoPackages."
            ),
        },
        {
            "table_name": "source_qa",
            "role": "source_lineage_qa",
            "spatial": False,
            "proposed_gpkg_type": "attributes",
            "status": "add",
            "description": (
                "Merged QA lineage copied from the four precursor "
                "Silver GeoPackages."
            ),
        },
    ]
)


print("Proposed final GeoPackage content contract")
print("------------------------------------------")

display(proposed_contents)

Proposed final GeoPackage content contract
------------------------------------------


,table_name,role,spatial,proposed_gpkg_type,status,description
0,storage_units,canonical_domain,False,NaN,review,Logical geological storage units independent o...
1,storage_features,canonical_domain,True,features,retain,Spatial representations of geological storage ...
2,storage_assessments,canonical_domain,False,NaN,review,Quantitative and qualitative assessments attac...
3,administrative_features,canonical_domain,True,features,retain,Regulatory and administrative carbon sequestra...
4,metadata_canada_geological_storage_unified,dataset_metadata,False,attributes,add,"Authoritative dataset-level provenance, interp..."
5,qa_canada_geological_storage_unified,dataset_qa,False,attributes,add,Authoritative quality-assurance summary for th...
6,source_metadata,source_lineage,False,attributes,add,Merged authoritative metadata lineage copied f...
7,source_qa,source_lineage_qa,False,attributes,add,Merged QA lineage copied from the four precurs...


In [20]:
# ---------------------------------------------------------------------------
# Finalize proposed GeoPackage content contract
# ---------------------------------------------------------------------------

final_contents = proposed_contents.copy()

final_contents.loc[
    final_contents["table_name"] == "storage_units",
    ["proposed_gpkg_type", "status"],
] = [
    "attributes",
    "revise",
]

final_contents.loc[
    final_contents["table_name"] == "storage_assessments",
    ["proposed_gpkg_type", "status"],
] = [
    "attributes",
    "revise",
]


# ---------------------------------------------------------------------------
# Record structural requirements for revised attribute tables
# ---------------------------------------------------------------------------

attribute_table_requirements = pd.DataFrame(
    [
        {
            "table_name": "storage_units",
            "gpkg_type": "attributes",
            "rowid_column": "fid",
            "semantic_id": "storage_unit_id",
            "required_change": (
                "Add INTEGER PRIMARY KEY fid and register in gpkg_contents."
            ),
        },
        {
            "table_name": "storage_assessments",
            "gpkg_type": "attributes",
            "rowid_column": "fid",
            "semantic_id": "storage_assessment_id",
            "required_change": (
                "Add INTEGER PRIMARY KEY fid and register in gpkg_contents."
            ),
        },
    ]
)


print("Final proposed GeoPackage content contract")
print("------------------------------------------")

display(final_contents)


print("\nRequired canonical attribute-table revisions")
print("--------------------------------------------")

display(attribute_table_requirements)

Final proposed GeoPackage content contract
------------------------------------------


,table_name,role,spatial,proposed_gpkg_type,status,description
0,storage_units,canonical_domain,False,attributes,revise,Logical geological storage units independent o...
1,storage_features,canonical_domain,True,features,retain,Spatial representations of geological storage ...
2,storage_assessments,canonical_domain,False,attributes,revise,Quantitative and qualitative assessments attac...
3,administrative_features,canonical_domain,True,features,retain,Regulatory and administrative carbon sequestra...
4,metadata_canada_geological_storage_unified,dataset_metadata,False,attributes,add,"Authoritative dataset-level provenance, interp..."
5,qa_canada_geological_storage_unified,dataset_qa,False,attributes,add,Authoritative quality-assurance summary for th...
6,source_metadata,source_lineage,False,attributes,add,Merged authoritative metadata lineage copied f...
7,source_qa,source_lineage_qa,False,attributes,add,Merged QA lineage copied from the four precurs...



Required canonical attribute-table revisions
--------------------------------------------


,table_name,gpkg_type,rowid_column,semantic_id,required_change
0,storage_units,attributes,fid,storage_unit_id,Add INTEGER PRIMARY KEY fid and register in gp...
1,storage_assessments,attributes,fid,storage_assessment_id,Add INTEGER PRIMARY KEY fid and register in gp...


## Final GeoPackage content model

The unified Canadian geological storage GeoPackage will use a mixed
feature/attribute-table structure consistent with the GeoPackage specification
and the precursor Silver packages.

The canonical domain model will contain four primary tables:

- `storage_units` — non-spatial logical geological storage objects.
- `storage_features` — spatial representations of geological storage objects.
- `storage_assessments` — non-spatial quantitative or qualitative assessments
  associated with storage units and/or spatial features.
- `administrative_features` — spatial regulatory or tenure features maintained
  separately from geological storage objects.

The two spatial tables will remain registered GeoPackage `features` tables.

The two non-spatial canonical tables will be revised during the formal rebuild
to become registered GeoPackage `attributes` tables. Each will receive a
surrogate `fid INTEGER PRIMARY KEY` used only as the GeoPackage / SQLite row
identifier. Existing stable semantic identifiers such as `storage_unit_id` and
`storage_assessment_id` will remain the authoritative domain identifiers and
will not be replaced by `fid`.

The documentation layer will add four registered attribute tables:

- `metadata_canada_geological_storage_unified` — authoritative package-level
  metadata for the unified Canadian artifact.
- `qa_canada_geological_storage_unified` — authoritative QA for the unified
  persisted artifact.
- `source_metadata` — merged metadata lineage copied from the four precursor
  Silver GeoPackages.
- `source_qa` — merged precursor QA lineage.

This design preserves the separation between:

1. GeoPackage row identity (`fid`);
2. stable domain identity (`storage_unit_id`, `storage_feature_id`,
   `storage_assessment_id`, `administrative_feature_id`);
3. package-level metadata and QA;
4. source-level provenance and QA lineage.

The current Notebook 11 artifact is not rebuilt in place. The structural
migration and final CanCO₂Re-compliant filename will be implemented in the next
build notebook or production script.